# Clase 3 · Productor de eventos por etapas
Ejecutá este notebook manualmente; representa un productor externo y no forma parte del Job.
Cada llegada debe procesarse y validarse antes de publicar la siguiente.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")
dbutils.widgets.dropdown("batch_id", "stream_001", ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "05 - Archivo a publicar")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

## Publicar un archivo inmutable
001: eventos normales. 002: duplicado, atraso tolerado y dos problemas de calidad.
003: avance a 12:26. 004: evento a 12:02 que llega demasiado tarde y otro normal.
005: control a 12:45 para cerrar las últimas ventanas de negocio.
Todos los tiempos son del 12/03/2026, en UTC.

In [ ]:
batch_id = normalize_stream_batch(dbutils.widgets.get("batch_id"))
result = publish_stream_batch(dbutils, spark, config, batch_id)
print(result)
preview = build_stream_rows(config, batch_id)[:6]
display(spark.createDataFrame(preview))
print(f"Ahora ejecutá el Job con expected_batch_id={batch_id}.")

Si el archivo ya existe, reejecutá el Job usando sus checkpoints. No vuelvas a generar ni sobrescribas
ese archivo. Para el siguiente lote, cambiá `batch_id` aquí y `expected_batch_id` en la ejecución del Job.
`expected_batch_id` es una expectativa del control, no un filtro de ingesta.